# Bronze — Delta Lake particionado

Este notebook lê arquivos `ecommerce_enderecos.parquet` da RAW, adiciona apenas auditoria da Bronze e salva como Delta no container `squad1`, particionado por ano, mês, dia e hora de ingestão.

##  Imports e parâmetros

In [0]:
import pandas as pd
from io import BytesIO
from functools import reduce
from datetime import datetime

from pyspark.sql import functions as F

CONTAINER_RAW = "raw"
RAW_FOLDER = "real-time-data"
NOME_ARQUIVO = "ecommerce_enderecos.parquet"

SCHEMA_DESTINO = "squad1"
TABELA_BRONZE = "bronze_ecommerce_enderecos"
TABELA_BRONZE_FULL = f"{SCHEMA_DESTINO}.{TABELA_BRONZE}"

print("Container RAW:", CONTAINER_RAW)
print("Pasta RAW:", RAW_FOLDER)
print("Arquivo alvo:", NOME_ARQUIVO)
print("Tabela Delta Bronze:", TABELA_BRONZE_FULL)

## Recriar client do container RAW

In [0]:
file_system_client = service_client.get_file_system_client(
    file_system=CONTAINER_RAW
)

print("File system client recriado para o container:", CONTAINER_RAW)

## Localizar arquivos `ecommerce_enderecos.parquet` na RAW

In [0]:
arquivos_encontrados = [
    p.name
    for p in file_system_client.get_paths(path=RAW_FOLDER)
    if (not p.is_directory) and p.name.endswith(NOME_ARQUIVO)
]

print(f"Arquivos encontrados para {NOME_ARQUIVO}: {len(arquivos_encontrados)}")

for arquivo in arquivos_encontrados:
    print(arquivo)

if len(arquivos_encontrados) == 0:
    raise Exception(f"Nenhum arquivo {NOME_ARQUIVO} encontrado em {CONTAINER_RAW}/{RAW_FOLDER}")

## Criar schema `squad1` e identificar arquivos já processados

In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SCHEMA_DESTINO}")

def tabela_existe(nome_tabela: str) -> bool:
    try:
        return spark.catalog.tableExists(nome_tabela)
    except Exception:
        try:
            spark.table(nome_tabela).limit(1).count()
            return True
        except Exception:
            return False

if tabela_existe(TABELA_BRONZE_FULL):
    df_arquivos_processados = (
        spark.table(TABELA_BRONZE_FULL)
        .select("bronze_source_file")
        .where(F.col("bronze_source_file").isNotNull())
        .dropDuplicates()
    )

    arquivos_processados = [
        row["bronze_source_file"]
        for row in df_arquivos_processados.collect()
    ]
else:
    arquivos_processados = []

arquivos_processados_set = set(arquivos_processados)

arquivos_novos = [
    arquivo
    for arquivo in arquivos_encontrados
    if arquivo not in arquivos_processados_set
]

print(f"Arquivos já processados na Bronze: {len(arquivos_processados)}")
print(f"Arquivos novos para processar: {len(arquivos_novos)}")

for arquivo in arquivos_novos:
    print("NOVO:", arquivo)

## Ler apenas arquivos novos e adicionar auditoria Bronze

Nesta etapa não aplicamos regra de negócio nem limpeza. Apenas adicionamos colunas técnicas de auditoria.

In [0]:
dfs_spark = []

for arquivo in arquivos_novos:
    print(f"Lendo arquivo novo: {arquivo}")

    file_client = file_system_client.get_file_client(arquivo)
    conteudo = file_client.download_file().readall()

    # Leitura do parquet via SDK por restrição de ABFSS no Serverless.
    # Em seguida convertemos para Spark DataFrame para salvar em Delta.
    df_pandas = pd.read_parquet(BytesIO(conteudo))
    df_spark = spark.createDataFrame(df_pandas)

    df_spark = (
        df_spark
        .withColumn("bronze_ingested_at", F.current_timestamp())
        .withColumn("bronze_source_file", F.lit(arquivo))
        .withColumn("bronze_ingest_year", F.year(F.col("bronze_ingested_at")))
        .withColumn("bronze_ingest_month", F.month(F.col("bronze_ingested_at")))
        .withColumn("bronze_ingest_day", F.dayofmonth(F.col("bronze_ingested_at")))
        .withColumn("bronze_ingest_hour", F.hour(F.col("bronze_ingested_at")))
    )

    dfs_spark.append(df_spark)

if len(dfs_spark) == 0:
    print("Nenhum arquivo novo para processar. A Bronze não será alterada.")
    df_bronze_micro_lote = None
else:
    df_bronze_micro_lote = reduce(
        lambda df1, df2: df1.unionByName(df2, allowMissingColumns=True),
        dfs_spark
    )

    print("Registros no micro-lote Bronze:", df_bronze_micro_lote.count())
    display(df_bronze_micro_lote.limit(10))

## Salvar Bronze como Delta particionada por data/hora de ingestão

In [0]:
if df_bronze_micro_lote is not None:
    (
        df_bronze_micro_lote.write
        .format("delta")
        .mode("append")
        .option("mergeSchema", "true")
        .partitionBy(
            "bronze_ingest_year",
            "bronze_ingest_month",
            "bronze_ingest_day",
            "bronze_ingest_hour"
        )
        .saveAsTable(TABELA_BRONZE_FULL)
    )

    print(f"Carga Bronze gravada com sucesso na tabela Delta: {TABELA_BRONZE_FULL}")
else:
    print("Carga Bronze ignorada: não havia arquivos novos.")

## Validar que os dados foram salvos

In [0]:
if tabela_existe(TABELA_BRONZE_FULL):
    df_validacao_bronze = spark.table(TABELA_BRONZE_FULL)

    print("=" * 80)
    print("VALIDAÇÃO DA BRONZE")
    print("Tabela:", TABELA_BRONZE_FULL)
    print("Total de registros:", df_validacao_bronze.count())
    print("Total de arquivos distintos:", df_validacao_bronze.select("bronze_source_file").distinct().count())
    print("Data/hora da validação:", datetime.now())
    print("=" * 80)

    display(
        df_validacao_bronze
        .select("bronze_source_file", "bronze_ingested_at")
        .dropDuplicates()
        .orderBy(F.col("bronze_ingested_at").desc())
    )

    display(
        df_validacao_bronze
        .groupBy(
            "bronze_ingest_year",
            "bronze_ingest_month",
            "bronze_ingest_day",
            "bronze_ingest_hour"
        )
        .count()
        .orderBy(
            F.col("bronze_ingest_year").desc(),
            F.col("bronze_ingest_month").desc(),
            F.col("bronze_ingest_day").desc(),
            F.col("bronze_ingest_hour").desc()
        )
    )
else:
    print(f"Tabela {TABELA_BRONZE_FULL} ainda não existe.")

## Consulta rápida da tabela Bronze

In [0]:
spark.sql(f"SHOW TABLES IN {SCHEMA_DESTINO}").show(truncate=False)

display(
    spark.table(TABELA_BRONZE_FULL)
    .orderBy(F.col("bronze_ingested_at").desc())
    .limit(20)
)